# ISMB 2026 Tutorial — Part 4
# Application #1: Spatial Omics and Histopathology Analysis with Multimodal Foundation Models
## Loki & Thor downstream platforms

**Instructor:** Guangyu Wang

**Prerequisites:** Part 3 notebook (`Part3_OmiCLIP_Introduction.ipynb`).

---

## Learning objectives

By the end of this notebook you will be able to:

1. Run **Loki** for tissue alignment between two consecutive Visium sections.
2. Annotate Visium spots using (a) a bulk RNA-seq reference and (b) a marker-gene panel — entirely through OmiCLIP embeddings.
3. Decompose each spot into cell-type proportions using Loki's reference-based decomposition.
4. Perform image ↔ transcriptomics retrieval across a tissue atlas.
5. Use **Thor** to predict spatial transcriptomics gene expression directly from an H&E image (no Visium required).

---

## 0. From foundation model to applications

Both Loki and Thor are *thin* wrappers around the frozen OmiCLIP encoders from Part 3. The trick is what we plug into the shared latent space:

| Task                                | What we put into the latent space                              | What we read out                        |
|-------------------------------------|----------------------------------------------------------------|-----------------------------------------|
| Tissue alignment (Loki)             | Image embeddings of two sections                                | Per-spot correspondence / warp          |
| Bulk-RNA annotation (Loki)          | Expression embeddings of spots + embeddings of bulk references | Nearest reference per spot              |
| Marker-gene annotation (Loki)       | Synthetic "pseudo-expression" of each marker panel             | Cosine match to each spot               |
| Cell-type decomposition (Loki)      | Reference cell-type centroids                                  | Non-negative mixing weights per spot    |
| Image ↔ transcriptomics retrieval   | Image patch embeddings vs. spot expression embeddings          | Top-k cross-modal neighbours            |
| Gene expression from H&E (Thor)     | Patch embeddings only                                          | Predicted per-gene log-expression       |

Everything below uses the same OmiCLIP checkpoint we loaded in Part 3.


## 1. Setup


In [ ]:
# --- Install (uncomment on first run / Colab) ---
# !pip install -q git+https://github.com/GuangyuWangLab2021/Loki.git
# !pip install -q git+https://github.com/GuangyuWangLab2021/Thor.git
# !pip install -q scanpy anndata scikit-learn umap-learn matplotlib seaborn squidpy


In [ ]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import torch

import loki
import thor
from loki.models import load_omiclip

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = Path('../data')
print('Loki :', loki.__version__)
print('Thor :', thor.__version__)
print('Device:', DEVICE)


In [ ]:
# Shared OmiCLIP backbone (frozen)
model, preprocess_image, tokenize_expression = load_omiclip(name='omiclip-base', device=DEVICE, pretrained=True)
model.eval();


---
## 2. Tissue alignment with Loki

When the same tissue block is cut into consecutive sections, the Visium spots on section A do *not* line up with those on section B — there is always rotation, translation, slight deformation, and missing tissue. Loki uses OmiCLIP image embeddings as a **modality-invariant feature space** to find spot correspondences and recover the alignment.


In [ ]:
from loki.align import VisiumSection, align_sections

secA = VisiumSection.from_dir(DATA_DIR / 'visium_sectionA')
secB = VisiumSection.from_dir(DATA_DIR / 'visium_sectionB')
print(secA, secB)

result = align_sections(
    secA, secB,
    encoder=model,
    preprocess=preprocess_image,
    feature='image',          # or 'expression' or 'joint'
    transform='affine',       # 'rigid' | 'affine' | 'tps'
    refine_with_mnn=True,
)
print('Mean alignment error (px):', result.error_px)


In [ ]:
from loki.plotting import plot_alignment
fig = plot_alignment(secA, secB, result, color='region', figsize=(10, 5))
plt.show()


### Notes for users

* `feature='joint'` averages the image and expression embeddings — works best when the two sections come from the same patient.
* `feature='image'` is more robust across patients / batches because tissue morphology is more conserved than batch-affected expression.
* `transform='tps'` (thin-plate spline) is recommended for tissues with significant non-rigid deformation (e.g., tumor with necrosis).


---
## 3. Annotation by bulk RNA-seq reference

Loki ships a small wrapper that:

1. Takes a bulk RNA-seq matrix with one column per reference label (e.g., TCGA cancer subtypes, GTEx tissues).
2. Runs each column through the **expression encoder** to obtain a reference embedding per label.
3. Assigns each Visium spot to the closest reference label by cosine similarity.

Because everything happens in the shared latent space, the bulk reference does not need to share genes 1-to-1 with the Visium data — only the gene tokens.


In [ ]:
from loki.annotate import bulk_rna_annotate

adata = sc.read_h5ad(DATA_DIR / 'demo_visium/counts.h5ad')
bulk_ref = pd.read_csv(DATA_DIR / 'tcga_brca_subtypes_bulk.csv', index_col=0)  # genes × subtypes
print(bulk_ref.shape, bulk_ref.columns.tolist())

annot = bulk_rna_annotate(
    adata, bulk_ref,
    encoder=model,
    tokenize=tokenize_expression,
    return_scores=True,
)
adata.obs['bulk_label']  = annot.labels
adata.obs['bulk_score']  = annot.top_score
adata.obs['bulk_label'].value_counts()


In [ ]:
sc.pl.spatial(adata, color=['bulk_label', 'bulk_score'], spot_size=120, ncols=2, frameon=False)


---
## 4. Annotation by marker-gene panel

If no bulk reference is available, Loki can also use **lists of marker genes** per cell type. Internally each marker list is converted to a *pseudo-expression* token sequence, encoded once, then matched to spot embeddings exactly as above.


In [ ]:
marker_panels = {
    'Tumor epithelium' : ['ERBB2', 'EPCAM', 'KRT8', 'KRT18', 'CDH1'],
    'T cells'          : ['CD3D', 'CD3E', 'CD8A', 'CD4', 'GZMB'],
    'Stromal/CAF'      : ['COL1A1', 'COL3A1', 'FAP', 'ACTA2', 'PDGFRB'],
    'Endothelium'      : ['PECAM1', 'VWF', 'CDH5', 'CLDN5'],
    'Macrophages'      : ['CD68', 'CD163', 'C1QA', 'C1QB', 'AIF1'],
}

from loki.annotate import marker_annotate
annot = marker_annotate(adata, marker_panels, encoder=model, tokenize=tokenize_expression)
adata.obs['marker_label'] = annot.labels
sc.pl.spatial(adata, color='marker_label', spot_size=120, frameon=False)


---
## 5. Cell-type decomposition

A Visium spot is ~55 µm wide and typically contains 1–20 cells. Loki performs **reference-based decomposition** in the latent space:

The reference embeddings are obtained from a single-cell atlas; the weights are recovered by a fast non-negative least-squares solve.


In [ ]:
from loki.decompose import decompose_spots

sc_ref = sc.read_h5ad(DATA_DIR / 'sc_breast_reference.h5ad')   # single-cell ref with .obs['cell_type']
props = decompose_spots(
    adata, sc_ref,
    label_col='cell_type',
    encoder=model,
    tokenize=tokenize_expression,
    n_neighbours=30,
)
print(props.head())

for ct in props.columns[:4]:
    adata.obs[ct] = props[ct].values
sc.pl.spatial(adata, color=list(props.columns[:4]), spot_size=120, ncols=2, frameon=False, cmap='magma')


---
## 6. Image ↔ transcriptomics retrieval at slide scale

We already saw the per-spot version of this in Part 3. Loki exposes a slide-level helper that:

* takes a **query image patch** (a region of interest from any H&E slide, not necessarily Visium), and
* returns spots from a database of Visium slides whose **expression** is closest to the query.

This is the building block for "find me a transcriptomic profile that looks like this lesion".


In [ ]:
from loki.retrieve import build_spot_index, query_image
from PIL import Image

index = build_spot_index(
    h5ad_paths=[DATA_DIR / 'demo_visium/counts.h5ad'],
    image_paths=[DATA_DIR / 'demo_visium/tissue_image.tif'],
    spot_csvs   =[DATA_DIR / 'demo_visium/spots.csv'],
    encoder=model,
    preprocess=preprocess_image,
    tokenize=tokenize_expression,
)

query_img = Image.open(DATA_DIR / 'query_patches/invasive_carcinoma.png').convert('RGB')
hits = query_image(query_img, index, k=10)
hits[['slide', 'spot', 'similarity', 'region', 'top_genes']]


---
## 7. Predicting spatial gene expression from H&E with **Thor**

Thor turns OmiCLIP into a *molecular microscope*: given **only** an H&E whole-slide image, it predicts per-pixel (or per-tile) gene expression as if Visium had been run on that slide.

The mechanism is conceptually simple:

1. Tile the H&E slide into 224 × 224 patches.
2. Encode each patch with the OmiCLIP image encoder.
3. Map the patch embedding to a gene-expression vector with a lightweight head trained on millions of Visium spots.
4. Stitch the per-tile predictions back into a spatial map.

Thor exposes two heads:
* `linear`  — fastest, interpretable, good for highly variable genes.
* `mlp`     — 2-layer MLP, recommended default.


In [ ]:
from thor.predict import HEPredictor

predictor = HEPredictor(
    omiclip_model=model,
    preprocess=preprocess_image,
    head='mlp',
    gene_set='HALLMARK+brca_markers',   # or a list of HGNC symbols
    device=DEVICE,
)

wsi_path = DATA_DIR / 'demo_visium/tissue_image.tif'
expr_map = predictor.predict_wsi(
    wsi_path,
    tile_size=224,
    stride=112,
    background_threshold=0.92,
)
print('Output:', expr_map)            # x, y, gene → predicted log1p expression
print('Genes  :', expr_map.genes[:10])


In [ ]:
from thor.plotting import plot_gene_map
for gene in ['ERBB2', 'CD3D', 'COL1A1', 'MKI67']:
    plot_gene_map(expr_map, gene=gene, wsi=wsi_path, alpha=0.65)
    plt.title(f'Thor prediction: {gene}')
    plt.show()


### Validate Thor against held-out Visium spots

When a Visium ground-truth is available we can quantify the per-gene Pearson correlation between Thor predictions and the measured expression.


In [ ]:
from thor.eval import evaluate_against_visium
scores = evaluate_against_visium(expr_map, adata, gene_subset=None)
scores.sort_values('pearson', ascending=False).head(15)


In [ ]:
sns.histplot(scores['pearson'], bins=40)
plt.axvline(0, color='k', linestyle='--')
plt.xlabel('Pearson r (Thor predicted vs. Visium measured)')
plt.title(f"Median r = {scores['pearson'].median():.2f}  (n={len(scores)} genes)")
plt.show()


---
## 8. End-to-end mini-case: scoring a TCGA H&E slide

We close the tutorial by chaining everything together: take a TCGA-BRCA H&E whole-slide image (no Visium attached), produce a Thor expression map, run Loki's marker-gene annotation **on the predicted expression**, and visualize the inferred tissue compartments.


In [ ]:
tcga_wsi = DATA_DIR / 'tcga/TCGA-AR-A1AR_HE.svs'
expr_map = predictor.predict_wsi(tcga_wsi, tile_size=224, stride=224)

pseudo_ad = expr_map.to_anndata()                # one pseudo-spot per tile
annot = marker_annotate(pseudo_ad, marker_panels, encoder=model, tokenize=tokenize_expression)
pseudo_ad.obs['compartment'] = annot.labels

from thor.plotting import plot_anndata_on_wsi
plot_anndata_on_wsi(pseudo_ad, wsi=tcga_wsi, color='compartment', alpha=0.55)
plt.title('Inferred tissue compartments — TCGA slide, no Visium used')
plt.show()


---
## 9. Take-home messages

* OmiCLIP gives you a **single shared latent space** for tissue images and transcriptomes.
* Loki exploits that space for the everyday spatial-omics tasks: **alignment, annotation, decomposition, retrieval**.
* Thor goes one step further and **predicts molecular maps from H&E alone**, opening up archival pathology cohorts where no Visium data exists.
* All three components are open source — please ⭐ the repos and report issues:
  * Loki — https://github.com/GuangyuWangLab2021/Loki
  * Thor — https://github.com/GuangyuWangLab2021/Thor

---

### Suggested exercises

1. Re-run section 2 with `feature='joint'` and compare the alignment error.
2. Replace the marker panel in section 4 with one of your own and check how the annotation moves on the slide.
3. Try the `head='linear'` predictor in Thor and compare per-gene Pearson r against the MLP head.
4. Pick three TCGA slides from the supplied folder and produce *ERBB2* prediction maps; correlate them with the clinical HER2 status in the metadata CSV.
